# Sugarcane Yield Prediction — Model Comparison & Prediction

Loads the early-stage train/test splits and preprocessor saved by `01_EDA_and_Preprocessing.ipynb`,
trains 5 regression models, compares them, picks the best performer, and uses it to predict yield.

**Models compared:** Linear Regression, Ridge Regression, Decision Tree, Random Forest, Gradient Boosting
**Metrics:** RMSE, MAE, R² (5-fold cross-validation + held-out test set)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Load saved splits + preprocessor

In [ ]:
X_train = pd.read_csv('X_train_raw.csv')
X_test = pd.read_csv('X_test_raw.csv')
y_train = pd.read_csv('y_train.csv').iloc[:, 0]
y_test = pd.read_csv('y_test.csv').iloc[:, 0]
preprocessor = joblib.load('preprocessor.joblib')

print("Train:", X_train.shape, "Test:", X_test.shape)

## 2. Define candidate models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=8),
    'Random Forest': RandomForestRegressor(random_state=42, n_estimators=200, max_depth=10, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42, n_estimators=200, max_depth=3, learning_rate=0.05)
}

## 3. Train, cross-validate, and evaluate each model

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])

    cv_rmse = -cross_val_score(pipe, X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    rmse = mean_squared_error(y_test, preds) ** 0.5
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results.append({
        'Model': name,
        'CV_RMSE_mean': cv_rmse.mean(),
        'CV_RMSE_std': cv_rmse.std(),
        'Test_RMSE': rmse,
        'Test_MAE': mae,
        'Test_R2': r2
    })
    fitted_pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values('Test_RMSE').reset_index(drop=True)
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=results_df, x='Test_RMSE', y='Model', hue='Model', legend=False, ax=axes[0], palette='crest')
axes[0].set_title('Test RMSE (lower is better)')
sns.barplot(data=results_df, x='Test_MAE', y='Model', hue='Model', legend=False, ax=axes[1], palette='crest')
axes[1].set_title('Test MAE (lower is better)')
sns.barplot(data=results_df, x='Test_R2', y='Model', hue='Model', legend=False, ax=axes[2], palette='crest')
axes[2].set_title('Test R² (higher is better)')
plt.tight_layout()
plt.show()

## 4. Select the best model

In [ ]:
best_model_name = results_df.loc[0, 'Model']
best_pipeline = fitted_pipelines[best_model_name]

print(f"Best model: {best_model_name}")
print(results_df.iloc[0])

## 5. Predicted vs. actual + residuals (best model)

In [ ]:
best_preds = best_pipeline.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, best_preds, alpha=0.4, color='seagreen')
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
axes[0].plot(lims, lims, 'r--', label='Perfect prediction')
axes[0].set_xlabel('Actual Yield (Quintal/Acre)')
axes[0].set_ylabel('Predicted Yield (Quintal/Acre)')
axes[0].set_title(f'{best_model_name}: Predicted vs Actual')
axes[0].legend()

residuals = y_test - best_preds
sns.histplot(residuals, kde=True, ax=axes[1], color='steelblue')
axes[1].set_title('Residual distribution')
axes[1].set_xlabel('Actual - Predicted')

plt.tight_layout()
plt.show()

## 6. Feature importance / coefficients (best model)

In [ ]:
feature_names = best_pipeline.named_steps['preprocessor'].get_feature_names_out()
model_step = best_pipeline.named_steps['model']

if hasattr(model_step, 'feature_importances_'):
    importances = model_step.feature_importances_
    imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
    imp_df = imp_df.sort_values('importance', ascending=False).head(20)
    plt.figure(figsize=(9, 8))
    sns.barplot(data=imp_df, x='importance', y='feature', hue='feature', legend=False, palette='viridis')
    plt.title(f'Top 20 feature importances -- {best_model_name}')
    plt.tight_layout()
    plt.show()
elif hasattr(model_step, 'coef_'):
    coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': model_step.coef_})
    coef_df['abs_coef'] = coef_df['coefficient'].abs()
    coef_df = coef_df.sort_values('abs_coef', ascending=False).head(20)
    plt.figure(figsize=(9, 8))
    sns.barplot(data=coef_df, x='coefficient', y='feature', hue='feature', legend=False, palette='vlag')
    plt.title(f'Top 20 coefficients (by magnitude) -- {best_model_name}')
    plt.tight_layout()
    plt.show()
else:
    print("Model has no built-in importance/coefficient attribute.")

## 7. Save the best model

Saves the **entire pipeline** (preprocessing + model together), so predicting on new raw data later
requires no manual re-encoding.

In [ ]:
joblib.dump(best_pipeline, 'best_model_pipeline.joblib')
print(f"Saved best_model_pipeline.joblib -- model: {best_model_name}")

## 8. Predict yield for new (early-stage) data

Example: predicting yield for the first 5 rows of the test set, plus a template for a single
hand-entered "new field" prediction.

In [ ]:
sample = X_test.head(5).copy()
sample_preds = best_pipeline.predict(sample)

comparison = pd.DataFrame({
    'Actual_Yield': y_test.head(5).values,
    'Predicted_Yield': sample_preds.round(2)
})
comparison['Abs_Error'] = (comparison['Actual_Yield'] - comparison['Predicted_Yield']).abs().round(2)
comparison

In [ ]:
# Template: predict yield for a single new field using early-stage inputs only.
# Fill in real values before or at planting time -- copy the column list from X_train.columns.

new_field = pd.DataFrame([{
    'Month': 'Jun', 'Temp_Min_C': 28.0, 'Temp_Max_C': 38.0, 'Temp_Avg_C': 33.0,
    'Rainfall_Total_mm': 1500, 'Rainfall_Seasonal_mm': 700, 'Humidity_%': 70,
    'Solar_Radiation_MJ_m2_day': 22, 'Wind_Speed_kmph': 8, 'Evapotranspiration_mm_day': 5,
    'Dew_Point_C': 15, 'Heat_Stress_Days': 5, 'Frost_Days': 2,
    'Soil_Type': 'Loamy', 'Soil_pH': 7.0, 'EC': 1.0, 'Organic_Carbon_%': 1.0,
    'Soil_Moisture_%': 25, 'Sand_%': 35, 'Silt_%': 30, 'Clay_%': 25,
    'Soil_Depth_cm': 25, 'Water_Holding_Capacity_%': 150,
    'Nitrogen_kg_per_acre': 45, 'Phosphorus_kg_per_acre': 55, 'Potassium_kg_per_acre': 90,
    'Zinc_mg_per_kg': 3, 'Iron_mg_per_kg': 5, 'Copper_mg_per_kg': 2, 'Manganese_mg_per_kg': 3,
    'Sulfur_kg_per_acre': 18, 'Irrigation_Frequency_Level': 'Medium',
    'Water_Quantity_liters_per_acre': 1200, 'Irrigation_Method_Type': 'Drip',
    'Groundwater_Level_meters': 10, 'Water_Quality_Category': 'Good',
    'Seed_Rate': 9, 'Row_Spacing_cm': 100, 'Row_Gap_cm': 90, 'Plant_Density': 100000,
    'Intercropping': 'No', 'Weed_Control': 'Chemical', 'Soil_Condition_At_Planting': 'Good',
    'Planting_Method': 'Straight Row', 'Mulching': 'Yes', 'Drainage_Condition': 'Good',
    'Variety': 'CoJ64', 'Latitude': 29.9, 'Longitude': 78.1, 'Altitude_m': 300,
    'Tehsil': 'Haridwar', 'Agro_Cluster': 'Narsan_Cluster', 'Season': 'Kharif', 'Year': 2026,
    'Sunshine_Hours': 7.5
}])

predicted_yield = best_pipeline.predict(new_field)[0]
print(f"Predicted yield for the new field: {predicted_yield:.2f} Quintal/Acre")

### Summary

- Best model: see Section 4 output above (chosen by lowest test RMSE).
- Saved as `best_model_pipeline.joblib` — reload with `joblib.load(...)` and call `.predict(new_data_df)`
  where `new_data_df` has the same 55 early-stage columns as `X_train`.
- R² in the 0.35-0.45 range on this dataset means the early-stage features explain a moderate share
  of yield variance -- expected, since many important drivers (disease pressure, exact fertilizer program,
  in-season weather shocks) aren't knowable at planting time. This is the realistic ceiling for a genuinely
  *early* prediction model; a model using late-stage features would score higher but couldn't be used for
  early decision-making.
